In [35]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

In [36]:
df = pd.read_csv('bbc_data.csv')
df["labels"].unique()

array(['entertainment', 'business', 'sport', 'politics', 'tech'],
      dtype=object)

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   data    2225 non-null   object
 1   labels  2225 non-null   object
dtypes: object(2)
memory usage: 34.9+ KB


In [38]:
df.head()

,data,labels
0,Musicians to tackle US red tape Musicians gro...,entertainment
1,"U2s desire to be number one U2, who have won ...",entertainment
2,Rocker Doherty in on-stage fight Rock singer ...,entertainment
3,Snicket tops US box office chart The film ada...,entertainment
4,"Oceans Twelve raids box office Oceans Twelve,...",entertainment


In [39]:
df.describe()

,data,labels
count,2225,2225
unique,2126,5
top,Musical treatment for Capra film The classic ...,sport
freq,2,511


In [40]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [41]:
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

In [42]:
df.isnull()

,data,labels
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
...,...,...
2220,False,False
2221,False,False
2222,False,False
2223,False,False


In [43]:
df.isnull().sum()

,0
data,0
labels,0


In [44]:
df.isnull().sum().sum()

np.int64(0)

In [45]:
df['processed_content'] = df['data'].apply(preprocess_text)

df['processed_content'].head()

,processed_content
0,"[musicians, tackle, us, red, tape, musicians, ..."
1,"[desire, number, one, three, prestigious, gram..."
2,"[rocker, doherty, fight, rock, singer, pete, d..."
3,"[snicket, tops, us, box, office, chart, film, ..."
4,"[oceans, twelve, raids, box, office, oceans, t..."


In [46]:
bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(df['processed_content'].apply(' '.join))

In [47]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['processed_content'].apply(' '.join))

In [48]:
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, df['labels'], test_size=0.2, random_state=42)

nb_model1 = MultinomialNB()

nb_model1.fit(X_train_bow, y_train)

MultinomialNB()

In [49]:
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(X_tfidf, df['labels'], test_size=0.2, random_state=42)

nb_model2 = MultinomialNB()

nb_model2.fit(X_train_tfidf, y_train)

MultinomialNB()

In [52]:

y_pred_bow = nb_model1.predict(X_test_bow)
print("BoW Model Performance:\n", classification_report(y_test, y_pred_bow))

y_pred_tfidf = nb_model2.predict(X_test_tfidf)
print("TF-IDF Model Performance:\n", classification_report(y_test, y_pred_tfidf))

BoW Model Performance:
                precision    recall  f1-score   support

     business       0.98      0.96      0.97       103
entertainment       1.00      0.98      0.99        84
     politics       0.98      0.99      0.98        80
        sport       1.00      0.99      0.99        98
         tech       0.95      1.00      0.98        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445

TF-IDF Model Performance:
                precision    recall  f1-score   support

     business       0.96      0.99      0.98       103
entertainment       1.00      0.95      0.98        84
     politics       0.90      0.97      0.93        80
        sport       0.99      0.99      0.99        98
         tech       1.00      0.93      0.96        80

     accuracy                           0.97       445
    macro avg       0.97      0.97      0.97       445
 weighted

In [53]:
custom_text1 = "Artificial intelligence is revolutionizing the tech industry, with companies racing to develop the next big innovation."

print("Input text: ", custom_text1)

processed_custom_text = ' '.join(preprocess_text(custom_text1))

custom_text_bow = bow_vectorizer.transform([processed_custom_text])
custom_text_tfidf = tfidf_vectorizer.transform([processed_custom_text])

predicted_category_bow = nb_model1.predict(custom_text_bow)
print(f"Predicted Category (BoW): {predicted_category_bow[0]}")

predicted_category_tfidf = nb_model2.predict(custom_text_tfidf)
print(f"Predicted Category (TF-IDF): {predicted_category_tfidf[0]}")

Input text:  Artificial intelligence is revolutionizing the tech industry, with companies racing to develop the next big innovation.
Predicted Category (BoW): tech
Predicted Category (TF-IDF): tech
